[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [asyncpg and psycopg3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/asyncpg-and-psycopg3-deep-dive.html)

# Server-Side Cursors


## What you will be able to do

Say where the rows are after `execute` returns, which is not where most people think. Measure what
each way of reading them costs the process, and say why fetching in batches from an ordinary cursor
saves less than it looks like it should. Open a named cursor, set how many rows it fetches at a
time, and know the two rules that decide whether it exists at all. Use `stream` for one pass with no
cursor to declare. And do the same in asyncpg, where the same rule about transactions is enforced
with a different message.


## The idea

### The problem

`cur.execute(...)` returns, and the natural reading is that the query has started. It has finished.
For an ordinary cursor, psycopg asks libpq for the result, and libpq does not return until the
server has sent every row. `fetchall` then turns bytes that are already in your process into Python
objects.

So the obvious economy is not one. Swapping `fetchall` for `fetchmany` in a loop saves the Python
list, which is real, and does nothing at all about the copy libpq is already holding. On a result
that does not fit in memory, both fail, at the same line.

### What a server-side cursor is

A cursor the server keeps. `DECLARE` makes it, `FETCH` asks for a batch, and the rows that have not
been asked for stay on the server. psycopg spells it `conn.cursor(name="something")`, and the name
is the whole difference: a cursor with a name is declared on the server, and one without is not.

Because the cursor lives on the server, it lives inside a transaction, which is where both of this
notebook's loud errors come from.

### Why it works that way

The protocol has two ways to run a query. The simple one sends a result in full. The extended one
can declare a portal and fetch from it in batches. libpq's ordinary path uses the first, because for
almost every query it is faster and simpler: one round trip, and the client can index the result
freely.

That choice is right up to the point where the result stops fitting, and then it is completely wrong,
which is why the fix is a different kind of cursor rather than a different way of reading one.

### Where this shows up

Exports, migrations, nightly jobs, and anything that says "select everything and process it". It is
invisible in development against a small table, and it takes the process out on the real one.

### What this notebook covers

Where the rows are, shown by asking the cursor before fetching anything. What four ways of reading
cost, measured. The named cursor and `itersize`. `stream`. asyncpg's version. Then the four
failures, one of which is silent and is the whole subject.

The row count at which the ordinary cursor would exhaust a free runtime is worth stating rather than
reaching: a few million rows of a handful of columns is enough, and a notebook that got there would
take the PostgreSQL you just installed down with it.

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import psycopg

with psycopg.connect("dbname=guide") as conn:
    with conn.cursor() as cur:                              # an ordinary cursor
        cur.execute("SELECT n FROM generate_series(1, 100000) AS n")
        print("ordinary cursor, after execute and before any fetch:")
        print("  rowcount:", cur.rowcount, "<- the server has already sent every row")

    with conn.cursor(name="batch") as cur:                  # a named cursor
        cur.execute("SELECT n FROM generate_series(1, 100000) AS n")
        print("named cursor, after execute and before any fetch:")
        print("  rowcount:", cur.rowcount, "<- nothing has been sent yet")
        print("  and it still reads them all:", sum(1 for _ in cur))
```

```
ordinary cursor, after execute and before any fetch:
  rowcount: 100000 <- the server has already sent every row
named cursor, after execute and before any fetch:
  rowcount: -1 <- nothing has been sent yet
  and it still reads them all: 100000
```

`rowcount` can only say a hundred thousand because the rows are already in the process. The named
cursor cannot say anything, because at that moment the server has been told to declare a cursor and
nothing else. Both loops end up with the same hundred thousand rows.


## Setup

Ten imports, both drivers, the server, and one helper that measures memory.

- `psycopg` and `asyncpg` are the drivers, and `errors` is the exception classes
- `subprocess` and `sys` run the measuring script, as well as installing the drivers with `version`
  and `PackageNotFoundError`
- `os`, `getpass` and `time` stand the server up, which **A Server of Your Own** takes apart

`grew_by` is the measurement this notebook rests on. It runs one way of reading three hundred
thousand rows in a Python of its own and reports how much that process grew, because resident memory
is a high-water mark that never falls: measuring two ways in one process would report the larger of
them twice. The number is rounded to the nearest twenty five megabytes, which is coarse on purpose,
because the exact figure moves between runs and between machines. What does not move is the shape of
the four answers, and that shape is the whole notebook.


In [1]:
import getpass
import os
import subprocess
import sys
import time
from importlib.metadata import PackageNotFoundError, version

try:
    if version("psycopg") < "3.3" or version("asyncpg") < "0.31":
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "psycopg[binary,pool]==3.3.6", "psycopg-pool==3.3.2", "asyncpg==0.31.0"],
                   check=True)

import asyncpg
import psycopg
from psycopg import errors

def shell(command):
    """Run a shell command and hand back what it printed, without letting it stop the notebook."""
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    return done.returncode, (done.stdout + done.stderr).strip()


def answering(database="postgres"):
    """Whether a server is there, asked the only way that needs no client binaries."""
    try:
        with psycopg.connect(f"dbname={database}", connect_timeout=2):
            return True
    except psycopg.OperationalError:
        return False


def start_server(wait=60):
    """Install and start PostgreSQL if nothing is answering. Returns what it had to do."""
    if answering():
        return "already running"
    if sys.platform != "linux":
        raise RuntimeError("No PostgreSQL is answering. Start your own server and run this again: "
                           "this cell only installs one on Linux, which is what Colab runs.")

    sudo = "" if os.geteuid() == 0 else "sudo "
    shell(f"{sudo}apt-get -qq update")
    shell(f"{sudo}apt-get -qq -y install postgresql postgresql-contrib")
    shell(f"{sudo}service postgresql start")                        # Colab has no systemd

    for attempt in range(1, wait + 1):                              # start returns before it listens
        if shell("pg_isready -q")[0] == 0:
            break
        print(f"  waiting for the cluster ({attempt})")              # a silent minute looks hung
        time.sleep(1)
    else:
        raise RuntimeError(f"PostgreSQL did not accept connections within {wait} seconds.")

    me = getpass.getuser()                                          # peer authentication wants a role
    asking = f"""sudo -u postgres psql -tAc "SELECT 1 FROM pg_roles WHERE rolname='{me}'" """
    if shell(asking)[1] != "1":                                     # named for the operating system user
        shell(f"sudo -u postgres createuser -s {me}")
    return "installed and started"

def build(rows=5000):
    """Make the guide database and its events table, and fill it once."""
    with psycopg.connect("dbname=postgres", autocommit=True) as conn:
        if not conn.execute("SELECT 1 FROM pg_database WHERE datname = 'guide'").fetchone():
            conn.execute("CREATE DATABASE guide")                   # cannot run in a transaction

    with psycopg.connect("dbname=guide", autocommit=True) as conn:
        for (leftover,) in conn.execute(                            # whatever an earlier run made
                "SELECT tablename FROM pg_tables "
                "WHERE schemaname = 'public' AND tablename <> 'events'").fetchall():
            conn.execute(f'DROP TABLE IF EXISTS "{leftover}" CASCADE')

        conn.execute("""CREATE TABLE IF NOT EXISTS events (
                            id bigserial PRIMARY KEY,
                            ts timestamptz NOT NULL DEFAULT now(),
                            kind text NOT NULL,
                            payload jsonb NOT NULL)""")
        if conn.execute("SELECT count(*) FROM events").fetchone()[0] == 0:
            conn.execute("""INSERT INTO events (kind, payload)
                            SELECT (ARRAY['click', 'view', 'purchase'])[1 + n %% 3],
                                   jsonb_build_object('n', n, 'size', 1 + n %% 7)
                            FROM generate_series(1, %s) AS n""", (rows,))
        return conn.execute("SELECT count(*) FROM events").fetchone()[0]

def report():
    """One line naming what this notebook is running against."""
    rows = build()                                                  # makes the database if it is new
    with psycopg.connect("dbname=guide") as conn:
        major = int(conn.execute("SHOW server_version_num").fetchone()[0]) // 10000
    return (f"PostgreSQL {major} | psycopg {version('psycopg')} | asyncpg {version('asyncpg')} "
            f"| events: {rows} rows")

MEASURING = """
import resource, sys, psycopg

def resident():
    raw = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss     # bytes on macOS, kilobytes on Linux
    return (raw if sys.platform == "darwin" else raw * 1024) / 1_000_000

QUERY = "SELECT n, repeat('x', 20) FROM generate_series(1, ROWS) AS n"
before = resident()

with psycopg.connect("dbname=guide") as conn:
    if "KIND" == "fetchall":
        with conn.cursor() as cur:
            cur.execute(QUERY)
            cur.fetchall()
    elif "KIND" == "fetchmany":
        with conn.cursor() as cur:
            cur.execute(QUERY)
            while cur.fetchmany(1000):
                pass
    elif "KIND" == "server":
        with conn.cursor(name="batch") as cur:
            cur.itersize = 1000
            cur.execute(QUERY)
            for _ in cur:
                pass
    elif "KIND" == "stream":
        with conn.cursor() as cur:
            for _ in cur.stream(QUERY):
                pass

print(round((resident() - before) / 25) * 25)                    # coarse, so two runs agree
"""


def grew_by(kind, rows=300_000):
    """How much the process grew reading the rows this way, in megabytes.

    Each way is measured in a Python of its own, because resident memory is a high-water mark that
    never falls, so measuring two ways in one process would report the larger of them twice. The
    answer is rounded to the nearest twenty five, because the exact number depends on the machine
    and on the allocator, and what this notebook is about is the difference between the four.
    """
    script = MEASURING.replace("KIND", kind).replace("ROWS", str(rows))
    done = subprocess.run([sys.executable, "-c", script], capture_output=True, text=True)
    return int(done.stdout.strip())


print("server:", start_server())
print(report())


server: already running
PostgreSQL 16 | psycopg 3.3.6 | asyncpg 0.31.0 | events: 5000 rows


## Worked examples

### Where the rows are

The first look asked `rowcount`. Here is the same question asked of the server, which knows whether
a cursor exists:


In [2]:
with psycopg.connect("dbname=guide") as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT n FROM generate_series(1, 1000) AS n")
        declared = conn.execute("SELECT count(*) FROM pg_cursors").fetchone()[0]
        print("ordinary cursor -> cursors on the server:", declared, "| rowcount:", cur.rowcount)

    with conn.cursor(name="batch") as cur:
        cur.execute("SELECT n FROM generate_series(1, 1000) AS n")
        declared = conn.execute("SELECT count(*) FROM pg_cursors").fetchone()[0]
        print("named cursor    -> cursors on the server:", declared, "| rowcount:", cur.rowcount)
        print("what it is called:",
              conn.execute("SELECT name FROM pg_cursors").fetchone()[0])


ordinary cursor -> cursors on the server: 0 | rowcount: 1000
named cursor    -> cursors on the server: 1 | rowcount: -1
what it is called: batch


`pg_cursors` is a view listing the cursors the server currently holds. The ordinary cursor is not in
it, because there is nothing on the server to hold: the rows have already been sent and the statement
is finished.

### What each way costs

Four ways of reading three hundred thousand rows, each in a process of its own:


In [3]:
for kind, description in (("fetchall", "execute, then fetchall()"),
                          ("fetchmany", "execute, then fetchmany(1000) in a loop"),
                          ("server", "a named cursor, itersize 1000"),
                          ("stream", "cur.stream(), one pass")):
    print(f"  {description:<38} grew the process by about {grew_by(kind):>3} MB")


  execute, then fetchall()               grew the process by about  75 MB
  execute, then fetchmany(1000) in a loop grew the process by about  25 MB
  a named cursor, itersize 1000          grew the process by about   0 MB
  cur.stream(), one pass                 grew the process by about   0 MB


Read the middle two together, because that pair is the point of this notebook.

`fetchall` pays twice: libpq holds the whole result, and then a Python list of tuples is built from
it. `fetchmany` in a loop pays once: no big list is built, so the number drops, and it does not drop
to nothing, because libpq is still holding every row. The saving is real and it is not the saving
people think they are making.

The named cursor pays for one batch at a time, which is why its number is close to nothing however
many rows there are. `stream` is the same, without declaring anything.

### The named cursor, and how big its batches are

`itersize` is how many rows each `FETCH` asks for:


In [4]:
with psycopg.connect("dbname=guide") as conn:
    with conn.cursor(name="small") as cur:
        cur.itersize = 25
        cur.execute("SELECT n FROM generate_series(1, 100) AS n")
        print("itersize 25 over 100 rows:", sum(1 for _ in cur), "rows read")

    with conn.cursor(name="one_batch") as cur:
        cur.itersize = 1000
        cur.execute("SELECT n FROM generate_series(1, 100) AS n")
        print("itersize 1000 over 100 rows:", sum(1 for _ in cur), "rows read")


itersize 25 over 100 rows: 100 rows read
itersize 1000 over 100 rows: 100 rows read


Both read every row and neither loop mentions batches, which is the point: `itersize` decides how
many rows cross the socket at a time and nothing about how you write the loop. The default is a
thousand, which is a reasonable answer to the trade between round trips and memory.

`fetchmany` on a named cursor is a real batch, and so is the batch size:


In [5]:
with psycopg.connect("dbname=guide") as conn:
    with conn.cursor(name="batches") as cur:
        cur.execute("SELECT n FROM generate_series(1, 10) AS n")
        while batch := cur.fetchmany(4):
            print("  a batch of", len(batch), ":", [row[0] for row in batch])


  a batch of 4 : [1, 2, 3, 4]
  a batch of 4 : [5, 6, 7, 8]
  a batch of 2 : [9, 10]


### stream, for one pass

`stream` is a generator over the rows with no cursor declared on the server. It is the thing to reach
for when you want to read a large result once, from beginning to end:


In [6]:
with psycopg.connect("dbname=guide") as conn:
    with conn.cursor() as cur:
        total = 0
        for row in cur.stream("SELECT n FROM generate_series(1, 100000) AS n"):
            total += row[0]

    print("summed every row:", total)
    print("cursors declared on the server:",
          conn.execute("SELECT count(*) FROM pg_cursors").fetchone()[0])
    print("the transaction is fine:", conn.info.transaction_status.name)


summed every row: 5000050000
cursors declared on the server: 0
the transaction is fine: INTRANS


Nothing was declared, and the process never held more than a row at a time.

"From beginning to end" is not a style preference. A `stream` puts the connection into a mode where
the server is sending rows and nothing else can be asked of it, and leaving the loop early does not
take it out of that mode: the transaction is left aborted and needs a rollback before the connection
can be used again. That is the third of the Common errors, and it is the reason to reach for a named
cursor rather than a stream whenever stopping early is a possibility.

### asyncpg

The same idea, and the same rule about transactions, with the syntax asyncpg uses:


In [7]:
conn = await asyncpg.connect(database="guide")

try:
    async for _ in conn.cursor("SELECT n FROM generate_series(1, 10) AS n"):
        pass
except asyncpg.exceptions.NoActiveSQLTransactionError as error:
    print("outside a transaction:", type(error).__name__ + ":", error)

async with conn.transaction():
    rows = [row[0] async for row in conn.cursor("SELECT n FROM generate_series(1, 5) AS n")]
print("inside one:          ", rows)


outside a transaction: NoActiveSQLTransactionError: cursor cannot be created outside of a transaction
inside one:           [1, 2, 3, 4, 5]


`conn.fetch` is asyncpg's ordinary path and brings everything back, exactly as psycopg's ordinary
cursor does. `conn.cursor` is the server-side one, and it must be inside `async with
conn.transaction()` for the same reason psycopg's named cursor must: the cursor lives on the server
and the transaction is what it lives in.


In [8]:
async with conn.transaction():
    cursor = await conn.cursor("SELECT n FROM generate_series(1, 100) AS n")
    print("fetch(3):", [row[0] for row in await cursor.fetch(3)])
    print("fetch(3):", [row[0] for row in await cursor.fetch(3)])
await conn.close()


fetch(3): [1, 2, 3]
fetch(3): [4, 5, 6]


### When to reach for which

| The result | What to use |
|---|---|
| small, and you want it all | `cur.execute(...)` then `fetchall()` |
| small, one row | `fetchone()`, or `scalar_row` |
| large, one pass, nothing fancy | `cur.stream(query)` |
| large, and you want batches | `conn.cursor(name=...)` with `itersize` |
| large, in asyncpg | `conn.cursor(...)` inside `async with conn.transaction()` |
| large, and you are about to write each row somewhere | a named cursor, and **COPY** for the writing |
| you only need a few rows | `LIMIT`, which beats every row of this table |

The default is an ordinary cursor. Reach for a named one when the result is large enough that
holding it would matter, and remember `LIMIT` first: the cheapest way to not hold a million rows is
not to ask for them.

### An export that does not grow, finished

Everything above, as the thing it is for: reading every row of a table and writing it somewhere,
with the process staying the same size however many rows there are.


In [9]:
def export(path, rows=200_000, batch=5_000):
    """Write every row to a file, a batch at a time, holding one batch at a time."""
    written = 0
    with psycopg.connect("dbname=guide") as conn:
        with conn.cursor(name="export") as cur:
            cur.itersize = batch
            cur.execute("SELECT n, repeat('x', 20) FROM generate_series(1, %s) AS n", (rows,))
            with open(path, "w") as out:
                for number, text in cur:
                    out.write(f"{number},{text}\n")
                    written += 1
    return written


import tempfile
target = os.path.join(tempfile.mkdtemp(), "export.csv")
print("rows written:", export(target))
print("file size on disk:", os.path.getsize(target) > 0)
print("and the reading process held one batch at a time, not", 200_000, "rows")


rows written: 200000
file size on disk: True
and the reading process held one batch at a time, not 200000 rows


The loop reads `for number, text in cur`, which looks exactly like the loop over an ordinary cursor
and behaves completely differently underneath. That is the whole ergonomic argument for named
cursors: the change is at the top, and the code that uses the rows does not move.

**COPY** is the faster answer for this particular job.

### Where each part came from

| In the export | What it relies on | The section that showed it |
|---|---|---|
| `conn.cursor(name="export")` | a cursor the server holds | Where the rows are |
| `cur.itersize = batch` | how many rows cross at a time | The named cursor |
| the `for` loop over the cursor | batching that the loop cannot see | The named cursor |
| the surrounding `with psycopg.connect(...)` | the transaction the cursor needs | The idea |
| `%s` for the row count | a value, not a formatted string | **Placeholders and Identifiers** |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/asyncpg-and-psycopg3-deep-dive/07-server-side-cursors-solutions.ipynb).

**1.** Run a query on an ordinary cursor and on a named one, and print `rowcount` for each before
fetching anything.


In [10]:
# your code here


**2.** Show that a named cursor appears in `pg_cursors` and an ordinary one does not.


In [11]:
# your code here


**3.** Measure how much the process grows reading two hundred thousand rows with `fetchall` and with
a named cursor.


In [12]:
# your code here


**4.** Read ten rows from a named cursor in batches of three and print each batch.


In [13]:
# your code here


**5.** Use `stream` to read the first five rows of a large query and stop, then show nothing was left
on the server.


In [14]:
# your code here


**6.** Read five rows through an asyncpg cursor, inside whatever it needs to be inside.


In [15]:
# your code here


## Common errors

### psycopg.errors.NoActiveSqlTransaction: DECLARE CURSOR can only be used in transaction blocks


In [16]:
with psycopg.connect("dbname=guide", autocommit=True) as conn:
    with conn.cursor(name="batch") as cur:
        cur.execute("SELECT n FROM generate_series(1, 100) AS n")


NoActiveSqlTransaction: DECLARE CURSOR can only be used in transaction blocks

`autocommit=True` means every statement is its own transaction, and a cursor the server holds has to
outlive the statement that declared it. There is nowhere for it to live.

This catches people who set `autocommit` for a good reason elsewhere in the program and then reach
for a named cursor on the same connection. The answer is a transaction around it:


In [17]:
with psycopg.connect("dbname=guide", autocommit=True) as conn:
    with conn.transaction():                                        # one, on purpose
        with conn.cursor(name="batch") as cur:
            cur.execute("SELECT n FROM generate_series(1, 100) AS n")
            print("read:", sum(1 for _ in cur), "rows")


read: 100 rows


### psycopg.errors.InvalidCursorName: cursor "batch" does not exist


In [18]:
with psycopg.connect("dbname=guide") as conn:
    cur = conn.cursor(name="batch")
    cur.execute("SELECT n FROM generate_series(1, 100) AS n")
    print("first two:", cur.fetchmany(2))

    conn.commit()                                                   # the transaction ends here
    cur.fetchmany(2)


first two: [(1,), (2,)]


InvalidCursorName: cursor "batch" does not exist

The same rule as the error above, arriving from the other direction: the cursor was fine until the
transaction it lived in ended, and committing ended it. The cursor went with it, and the name no
longer refers to anything.

A named cursor is therefore bounded by its transaction, which is worth planning for in a long job:
commit between batches and the cursor is gone, so either the whole read is one transaction or the
job is written to restart from a key it remembers:


In [19]:
with psycopg.connect("dbname=guide") as conn:
    with conn.cursor(name="batch") as cur:
        cur.execute("SELECT n FROM generate_series(1, 100) AS n")
        print("all of it inside one transaction:", sum(1 for _ in cur), "rows")

last_seen = 0
with psycopg.connect("dbname=guide") as conn:                       # or remember where you were
    for _ in range(3):
        rows = conn.execute("SELECT n FROM generate_series(1, 100) AS n WHERE n > %s ORDER BY n "
                            "LIMIT 25", (last_seen,)).fetchall()
        last_seen = rows[-1][0]
        conn.commit()
    print("three committed batches, up to:", last_seen)


all of it inside one transaction: 100 rows
three committed batches, up to: 75


### psycopg.errors.InFailedSqlTransaction, from a stream that was left early


In [20]:
with psycopg.connect("dbname=guide") as conn:
    with conn.cursor() as cur:
        got = []
        for row in cur.stream("SELECT n FROM generate_series(1, 100000) AS n"):
            got.append(row[0])
            if len(got) == 5:
                break                                               # the other rows are still coming

    print("the rows arrived:", got)
    print("the transaction:  ", conn.info.transaction_status.name)
    conn.execute("SELECT 1")


the rows arrived: [1, 2, 3, 4, 5]
the transaction:   INERROR


InFailedSqlTransaction: current transaction is aborted, commands ignored until end of transaction block

The five rows are fine. The connection is not: `stream` puts it into a mode where the server is
sending a result one row at a time, and abandoning the loop leaves it there, so the next statement
gets the aborted-transaction error from **Transactions and Errors** instead of an answer.

Calling `close()` on the generator does not help either, which is worth knowing because it is the
obvious guess. Three things do:


In [21]:
print("a rollback clears it:")
with psycopg.connect("dbname=guide") as conn:
    with conn.cursor() as cur:
        for row in cur.stream("SELECT n FROM generate_series(1, 1000) AS n"):
            break
    conn.rollback()
    print("  ", conn.info.transaction_status.name, conn.execute("SELECT 1").fetchone())

print("a named cursor can simply stop:")
with psycopg.connect("dbname=guide") as conn:
    with conn.cursor(name="stoppable") as cur:
        cur.execute("SELECT n FROM generate_series(1, 100000) AS n")
        got = [row[0] for row in cur.fetchmany(5)]
    print("  ", got, conn.info.transaction_status.name)

print("or ask for fewer rows in the first place:")
with psycopg.connect("dbname=guide") as conn:
    print("  ", conn.execute("SELECT n FROM generate_series(1, 100000) AS n LIMIT 5").fetchall())


a rollback clears it:
   IDLE (1,)
a named cursor can simply stop:
   [1, 2, 3, 4, 5] INTRANS
or ask for fewer rows in the first place:
   [(1,), (2,), (3,), (4,), (5,)]


### No error, and a process the size of the table: fetchmany on an ordinary cursor


In [22]:
print("reading 300,000 rows:")
for kind, description in (("fetchall", "fetchall()"),
                          ("fetchmany", "fetchmany(1000) in a loop"),
                          ("server", "a named cursor")):
    print(f"  {description:<28} about {grew_by(kind):>3} MB")


reading 300,000 rows:
  fetchall()                   about  75 MB
  fetchmany(1000) in a loop    about  25 MB
  a named cursor               about   0 MB


Nothing raises, every version returns the same rows, and the second one looks like the fix. It is
not: the loop stopped building one big list, and libpq is still holding the entire result, which is
where most of the memory was.

The line between them is whether the rows are on the server or in your process, and only the name on
the cursor decides that. If the result is too big to hold, batching an ordinary cursor does not help,
and the version that does is one argument away:


In [23]:
with psycopg.connect("dbname=guide") as conn:
    with conn.cursor() as ordinary:
        ordinary.execute("SELECT n FROM generate_series(1, 1000) AS n")
        print("ordinary: rows are here already, rowcount =", ordinary.rowcount)

    with conn.cursor(name="held") as named:
        named.execute("SELECT n FROM generate_series(1, 1000) AS n")
        print("named:    rows are on the server, rowcount =", named.rowcount)


ordinary: rows are here already, rowcount = 1000
named:    rows are on the server, rowcount = -1


## Recap

- `execute` on an ordinary cursor does not start the query, it finishes it. libpq holds the whole
  result before `fetchall` is called, which is why `rowcount` can answer immediately.
- `fetchmany` on an ordinary cursor saves building one large Python list, and saves nothing on the
  copy libpq already holds. Measured, that is most of the memory but not all of it.
- A cursor with a name is declared on the server, appears in `pg_cursors`, and holds its rows there.
  `itersize` decides how many cross at a time.
- A named cursor lives inside a transaction. `autocommit` leaves it nowhere to live, and a commit
  destroys it.
- `stream` reads a large result in one pass with nothing declared, and has to be read to the end:
  leaving the loop early aborts the transaction, and a named cursor is what to use when stopping
  early is possible.
- asyncpg's `conn.cursor` is the same idea and must be inside `async with conn.transaction()`.
- `LIMIT` beats all of this when you only wanted a few rows.


## What is next

The **COPY** notebook is the fastest way in and out: bulk loading in both drivers, reading a whole
table out the same way, and the row with one column too many that stops a load partway.


---

&#8592; **Previous:** [JSONB](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/asyncpg-and-psycopg3-deep-dive/06-jsonb.ipynb)  &nbsp;·&nbsp;  [asyncpg and psycopg3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/asyncpg-and-psycopg3-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [COPY](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/asyncpg-and-psycopg3-deep-dive/08-copy.ipynb) &#8594;
